# Segment-Aware Data Pipeline: Points 1 and 2

This notebook implements the first two stages of the project using the actual structure of the dataset in `data/`:

1. Parse the raw sensor files, extract contiguous activity segments, and pair the four streams.
2. Generate fixed-rate, fixed-length fused windows from paired segments and write cache artifacts.

The key dataset-specific decision is that each file is an ordered concatenation of labeled activity segments, not one continuous globally synchronized recording. We therefore align by `(subject_id, activity_label, occurrence_index)` and use relative time within each segment.

## Defaults Chosen Here

- Target sampling rate: `20 Hz`
- Window length: `3.0 s`
- Window stride: `1.0 s`
- Samples per window: `60`
- Required streams: `phone/accel`, `phone/gyro`, `watch/accel`, `watch/gyro`
- Validation split: deterministic subject split from Kaggle `train/` subjects
- Artifacts written under `../artifacts/segments/` and `../artifacts/windows/`

The implementation is deliberately dependency-light so it can run in a bare Python environment.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
import csv
import json
import math
import pickle
import random
import statistics
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not locate the project root containing data/")

DATA_ROOT = PROJECT_ROOT / "data"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"

TARGET_HZ = 20
WINDOW_SECONDS = 3.0
WINDOW_STRIDE_SECONDS = 1.0
WINDOW_LENGTH = int(TARGET_HZ * WINDOW_SECONDS)
MIN_SEGMENT_SECONDS = WINDOW_SECONDS
VALIDATION_SEED = 42
VALIDATION_SUBJECT_COUNT = 4

REQUIRED_STREAMS: Tuple[Tuple[str, str], ...] = (
    ("phone", "accel"),
    ("phone", "gyro"),
    ("watch", "accel"),
    ("watch", "gyro"),
)

@dataclass(frozen=True)
class SensorFileInfo:
    split: str
    device: str
    sensor: str
    subject_id: int
    path: str


@dataclass(frozen=True)
class RawRow:
    subject_id: int
    activity_label: str
    timestamp_ns: int
    x: float
    y: float
    z: float


@dataclass
class StreamData:
    info: SensorFileInfo
    labels: List[str]
    timestamps_ns: List[int]
    values_xyz: List[Tuple[float, float, float]]


@dataclass(frozen=True)
class SegmentRecord:
    split: str
    subject_id: int
    device: str
    sensor: str
    activity_label: str
    occurrence_index: int
    segment_index_global: int
    start_row: int
    end_row_exclusive: int
    n_rows: int
    start_ts_ns: int
    end_ts_ns: int
    duration_s: float
    native_hz: Optional[float]
    has_non_monotonic_timestamp: bool
    is_valid: bool
    drop_reason: str
    path: str


@dataclass(frozen=True)
class PairedSegmentRecord:
    split: str
    subject_id: int
    activity_label: str
    occurrence_index: int
    common_duration_s: float
    is_valid_four_stream_pair: bool
    drop_reason: str
    stream_presence: str
    phone_accel_start_row: int
    phone_accel_end_row_exclusive: int
    phone_accel_duration_s: float
    phone_accel_native_hz: Optional[float]
    phone_gyro_start_row: int
    phone_gyro_end_row_exclusive: int
    phone_gyro_duration_s: float
    phone_gyro_native_hz: Optional[float]
    watch_accel_start_row: int
    watch_accel_end_row_exclusive: int
    watch_accel_duration_s: float
    watch_accel_native_hz: Optional[float]
    watch_gyro_start_row: int
    watch_gyro_end_row_exclusive: int
    watch_gyro_duration_s: float
    watch_gyro_native_hz: Optional[float]


print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Artifacts: {ARTIFACT_ROOT.resolve()}")
print(f"Window config: {WINDOW_SECONDS:.1f}s @ {TARGET_HZ} Hz -> {WINDOW_LENGTH} samples")

In [ ]:
def parse_sensor_filename(path: Path) -> SensorFileInfo:
    parts = path.parts
    try:
        split = parts[-4]
        device = parts[-3]
        sensor = parts[-2]
    except IndexError as exc:
        raise ValueError(f"Unexpected path layout: {path}") from exc

    stem_parts = path.stem.split("_")
    if len(stem_parts) != 4 or stem_parts[0] != "data":
        raise ValueError(f"Unexpected filename format: {path.name}")

    subject_id = int(stem_parts[1])
    if stem_parts[2] != sensor or stem_parts[3] != device:
        raise ValueError(f"Filename stream mismatch for {path.name}")

    return SensorFileInfo(
        split=split,
        device=device,
        sensor=sensor,
        subject_id=subject_id,
        path=str(path),
    )


def iter_sensor_rows(path: Path) -> Iterator[RawRow]:
    info = parse_sensor_filename(path)
    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue
            if line.endswith(";"):
                line = line[:-1]
            fields = line.split(",")
            if len(fields) != 6:
                raise ValueError(f"{path}:{line_number} expected 6 fields, found {len(fields)}")

            row = RawRow(
                subject_id=int(fields[0]),
                activity_label=fields[1],
                timestamp_ns=int(fields[2]),
                x=float(fields[3]),
                y=float(fields[4]),
                z=float(fields[5]),
            )
            if row.subject_id != info.subject_id:
                raise ValueError(
                    f"{path}:{line_number} subject id {row.subject_id} does not match filename {info.subject_id}"
                )
            yield row


def load_stream(path: Path) -> StreamData:
    info = parse_sensor_filename(path)
    labels: List[str] = []
    timestamps_ns: List[int] = []
    values_xyz: List[Tuple[float, float, float]] = []

    for row in iter_sensor_rows(path):
        labels.append(row.activity_label)
        timestamps_ns.append(row.timestamp_ns)
        values_xyz.append((row.x, row.y, row.z))

    if not labels:
        raise ValueError(f"Empty stream file: {path}")

    return StreamData(
        info=info,
        labels=labels,
        timestamps_ns=timestamps_ns,
        values_xyz=values_xyz,
    )


def infer_native_hz(timestamps_ns: Sequence[int]) -> Optional[float]:
    positive_diffs = [
        next_ts - current_ts
        for current_ts, next_ts in zip(timestamps_ns, timestamps_ns[1:])
        if next_ts > current_ts
    ]
    if not positive_diffs:
        return None
    median_dt_ns = statistics.median(positive_diffs)
    if median_dt_ns <= 0:
        return None
    return 1_000_000_000.0 / median_dt_ns


def extract_segments(stream: StreamData) -> List[SegmentRecord]:
    labels = stream.labels
    timestamps_ns = stream.timestamps_ns
    segments: List[SegmentRecord] = []
    occurrence_counter: Dict[str, int] = defaultdict(int)

    start_row = 0
    current_label = labels[0]
    current_occurrence = occurrence_counter[current_label]
    segment_index_global = 0

    for row_index in range(1, len(labels) + 1):
        at_end = row_index == len(labels)
        label_changed = not at_end and labels[row_index] != current_label
        if not at_end and not label_changed:
            continue

        end_row_exclusive = row_index
        segment_timestamps = timestamps_ns[start_row:end_row_exclusive]
        start_ts_ns = segment_timestamps[0]
        end_ts_ns = segment_timestamps[-1]
        positive_diffs = [
            next_ts - current_ts
            for current_ts, next_ts in zip(segment_timestamps, segment_timestamps[1:])
            if next_ts > current_ts
        ]
        has_non_monotonic_timestamp = any(
            next_ts <= current_ts for current_ts, next_ts in zip(segment_timestamps, segment_timestamps[1:])
        )
        native_hz = infer_native_hz(segment_timestamps)
        duration_s = max(0.0, (end_ts_ns - start_ts_ns) / 1_000_000_000.0)

        is_valid = True
        drop_reason = ""
        if len(segment_timestamps) < 2:
            is_valid = False
            drop_reason = "fewer_than_two_rows"
        elif native_hz is None:
            is_valid = False
            drop_reason = "missing_positive_timestamp_deltas"
        elif has_non_monotonic_timestamp:
            is_valid = False
            drop_reason = "non_monotonic_timestamp_within_segment"

        segments.append(
            SegmentRecord(
                split=stream.info.split,
                subject_id=stream.info.subject_id,
                device=stream.info.device,
                sensor=stream.info.sensor,
                activity_label=current_label,
                occurrence_index=current_occurrence,
                segment_index_global=segment_index_global,
                start_row=start_row,
                end_row_exclusive=end_row_exclusive,
                n_rows=end_row_exclusive - start_row,
                start_ts_ns=start_ts_ns,
                end_ts_ns=end_ts_ns,
                duration_s=duration_s,
                native_hz=native_hz,
                has_non_monotonic_timestamp=has_non_monotonic_timestamp,
                is_valid=is_valid,
                drop_reason=drop_reason,
                path=stream.info.path,
            )
        )

        occurrence_counter[current_label] += 1
        segment_index_global += 1

        if not at_end:
            start_row = row_index
            current_label = labels[row_index]
            current_occurrence = occurrence_counter[current_label]

    return segments


def stream_key(device: str, sensor: str) -> str:
    return f"{device}_{sensor}"


def discover_subject_ids(data_root: Path) -> Dict[str, List[int]]:
    split_to_subject_ids: Dict[str, List[int]] = {}
    for split in ("train", "test"):
        subject_ids = []
        phone_accel_dir = data_root / split / "phone" / "accel"
        for path in sorted(phone_accel_dir.glob("*.txt")):
            subject_ids.append(parse_sensor_filename(path).subject_id)
        split_to_subject_ids[split] = sorted(subject_ids)
    return split_to_subject_ids


def load_subject_streams(data_root: Path, split: str, subject_id: int) -> Dict[Tuple[str, str], StreamData]:
    streams: Dict[Tuple[str, str], StreamData] = {}
    for device, sensor in REQUIRED_STREAMS:
        path = data_root / split / device / sensor / f"data_{subject_id}_{sensor}_{device}.txt"
        streams[(device, sensor)] = load_stream(path)
    return streams


In [ ]:
def pair_segments_for_subject(
    split: str,
    subject_id: int,
    segments_by_stream: Dict[Tuple[str, str], List[SegmentRecord]],
) -> List[PairedSegmentRecord]:
    keyed_segments: Dict[Tuple[str, str], Dict[Tuple[str, int], SegmentRecord]] = {}
    order_index: Dict[Tuple[str, int], int] = {}
    all_keys = set()

    for stream, segment_list in segments_by_stream.items():
        segment_map: Dict[Tuple[str, int], SegmentRecord] = {}
        keyed_segments[stream] = segment_map
        for segment in segment_list:
            key = (segment.activity_label, segment.occurrence_index)
            segment_map[key] = segment
            all_keys.add(key)
            existing_order = order_index.get(key)
            if existing_order is None or segment.segment_index_global < existing_order:
                order_index[key] = segment.segment_index_global

    paired_records: List[PairedSegmentRecord] = []
    ordered_keys = sorted(all_keys, key=lambda key: (order_index.get(key, 10**9), key[0], key[1]))

    for activity_label, occurrence_index in ordered_keys:
        segments: Dict[Tuple[str, str], Optional[SegmentRecord]] = {
            stream: keyed_segments.get(stream, {}).get((activity_label, occurrence_index))
            for stream in REQUIRED_STREAMS
        }
        present_streams = [stream_key(device, sensor) for (device, sensor), segment in segments.items() if segment]
        stream_presence = ",".join(sorted(present_streams))

        missing_streams = [stream_key(device, sensor) for (device, sensor), segment in segments.items() if segment is None]
        if missing_streams:
            paired_records.append(
                PairedSegmentRecord(
                    split=split,
                    subject_id=subject_id,
                    activity_label=activity_label,
                    occurrence_index=occurrence_index,
                    common_duration_s=0.0,
                    is_valid_four_stream_pair=False,
                    drop_reason=f"missing_streams:{'|'.join(missing_streams)}",
                    stream_presence=stream_presence,
                    phone_accel_start_row=segments[("phone", "accel")].start_row if segments[("phone", "accel")] else -1,
                    phone_accel_end_row_exclusive=segments[("phone", "accel")].end_row_exclusive if segments[("phone", "accel")] else -1,
                    phone_accel_duration_s=segments[("phone", "accel")].duration_s if segments[("phone", "accel")] else 0.0,
                    phone_accel_native_hz=segments[("phone", "accel")].native_hz if segments[("phone", "accel")] else None,
                    phone_gyro_start_row=segments[("phone", "gyro")].start_row if segments[("phone", "gyro")] else -1,
                    phone_gyro_end_row_exclusive=segments[("phone", "gyro")].end_row_exclusive if segments[("phone", "gyro")] else -1,
                    phone_gyro_duration_s=segments[("phone", "gyro")].duration_s if segments[("phone", "gyro")] else 0.0,
                    phone_gyro_native_hz=segments[("phone", "gyro")].native_hz if segments[("phone", "gyro")] else None,
                    watch_accel_start_row=segments[("watch", "accel")].start_row if segments[("watch", "accel")] else -1,
                    watch_accel_end_row_exclusive=segments[("watch", "accel")].end_row_exclusive if segments[("watch", "accel")] else -1,
                    watch_accel_duration_s=segments[("watch", "accel")].duration_s if segments[("watch", "accel")] else 0.0,
                    watch_accel_native_hz=segments[("watch", "accel")].native_hz if segments[("watch", "accel")] else None,
                    watch_gyro_start_row=segments[("watch", "gyro")].start_row if segments[("watch", "gyro")] else -1,
                    watch_gyro_end_row_exclusive=segments[("watch", "gyro")].end_row_exclusive if segments[("watch", "gyro")] else -1,
                    watch_gyro_duration_s=segments[("watch", "gyro")].duration_s if segments[("watch", "gyro")] else 0.0,
                    watch_gyro_native_hz=segments[("watch", "gyro")].native_hz if segments[("watch", "gyro")] else None,
                )
            )
            continue

        duration_values = [segment.duration_s for segment in segments.values() if segment is not None]
        common_duration_s = min(duration_values)
        invalid_segments = [segment for segment in segments.values() if segment and not segment.is_valid]
        drop_reason = ""
        is_valid = True

        if invalid_segments:
            is_valid = False
            invalid_reasons = sorted({segment.drop_reason for segment in invalid_segments})
            drop_reason = f"invalid_segment:{'|'.join(invalid_reasons)}"
        elif common_duration_s < MIN_SEGMENT_SECONDS:
            is_valid = False
            drop_reason = "segment_shorter_than_window"

        paired_records.append(
            PairedSegmentRecord(
                split=split,
                subject_id=subject_id,
                activity_label=activity_label,
                occurrence_index=occurrence_index,
                common_duration_s=common_duration_s,
                is_valid_four_stream_pair=is_valid,
                drop_reason=drop_reason,
                stream_presence=stream_presence,
                phone_accel_start_row=segments[("phone", "accel")].start_row,
                phone_accel_end_row_exclusive=segments[("phone", "accel")].end_row_exclusive,
                phone_accel_duration_s=segments[("phone", "accel")].duration_s,
                phone_accel_native_hz=segments[("phone", "accel")].native_hz,
                phone_gyro_start_row=segments[("phone", "gyro")].start_row,
                phone_gyro_end_row_exclusive=segments[("phone", "gyro")].end_row_exclusive,
                phone_gyro_duration_s=segments[("phone", "gyro")].duration_s,
                phone_gyro_native_hz=segments[("phone", "gyro")].native_hz,
                watch_accel_start_row=segments[("watch", "accel")].start_row,
                watch_accel_end_row_exclusive=segments[("watch", "accel")].end_row_exclusive,
                watch_accel_duration_s=segments[("watch", "accel")].duration_s,
                watch_accel_native_hz=segments[("watch", "accel")].native_hz,
                watch_gyro_start_row=segments[("watch", "gyro")].start_row,
                watch_gyro_end_row_exclusive=segments[("watch", "gyro")].end_row_exclusive,
                watch_gyro_duration_s=segments[("watch", "gyro")].duration_s,
                watch_gyro_native_hz=segments[("watch", "gyro")].native_hz,
            )
        )

    return paired_records


def write_csv_rows(path: Path, rows: List[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def build_segment_index(data_root: Path, artifact_root: Path) -> dict:
    subject_ids_by_split = discover_subject_ids(data_root)
    segment_rows: List[dict] = []
    paired_rows: List[dict] = []
    summary = {
        "subject_ids_by_split": subject_ids_by_split,
        "stream_segment_count": 0,
        "paired_segment_count": 0,
        "valid_four_stream_pair_count": 0,
        "dropped_pair_reasons": Counter(),
    }

    for split, subject_ids in subject_ids_by_split.items():
        for subject_id in subject_ids:
            subject_streams = load_subject_streams(data_root, split, subject_id)
            subject_segments = {
                stream: extract_segments(stream_data)
                for stream, stream_data in subject_streams.items()
            }

            for segment_list in subject_segments.values():
                for segment in segment_list:
                    segment_rows.append(asdict(segment))
                    summary["stream_segment_count"] += 1

            paired_segments = pair_segments_for_subject(split, subject_id, subject_segments)
            for paired_segment in paired_segments:
                paired_rows.append(asdict(paired_segment))
                summary["paired_segment_count"] += 1
                if paired_segment.is_valid_four_stream_pair:
                    summary["valid_four_stream_pair_count"] += 1
                else:
                    summary["dropped_pair_reasons"][paired_segment.drop_reason] += 1

    segment_dir = artifact_root / "segments"
    write_csv_rows(segment_dir / "stream_segments.csv", segment_rows)
    write_csv_rows(segment_dir / "paired_segments.csv", paired_rows)

    summary_payload = {
        "subject_ids_by_split": summary["subject_ids_by_split"],
        "stream_segment_count": summary["stream_segment_count"],
        "paired_segment_count": summary["paired_segment_count"],
        "valid_four_stream_pair_count": summary["valid_four_stream_pair_count"],
        "dropped_pair_reasons": dict(summary["dropped_pair_reasons"]),
    }
    (segment_dir / "segment_index_summary.json").write_text(
        json.dumps(summary_payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    return {
        "summary": summary_payload,
        "stream_segments": segment_rows,
        "paired_segments": paired_rows,
    }


segment_index = build_segment_index(DATA_ROOT, ARTIFACT_ROOT)
print(json.dumps(segment_index["summary"], indent=2))

In [ ]:
def build_dataset_subject_splits(data_root: Path) -> Dict[str, List[int]]:
    subject_ids_by_split = discover_subject_ids(data_root)
    train_subjects = list(subject_ids_by_split["train"])
    shuffled = train_subjects[:]
    random.Random(VALIDATION_SEED).shuffle(shuffled)
    val_subjects = sorted(shuffled[:VALIDATION_SUBJECT_COUNT])
    actual_train_subjects = sorted(subject_id for subject_id in train_subjects if subject_id not in val_subjects)
    test_subjects = list(subject_ids_by_split["test"])
    return {
        "train": actual_train_subjects,
        "val": val_subjects,
        "test": test_subjects,
    }


def map_dataset_split_to_raw_split(dataset_split: str) -> str:
    return "train" if dataset_split in {"train", "val"} else "test"


def extract_segment_times_and_values(stream: StreamData, segment: SegmentRecord) -> Tuple[List[float], List[Tuple[float, float, float]]]:
    timestamps_ns = stream.timestamps_ns[segment.start_row:segment.end_row_exclusive]
    values_xyz = stream.values_xyz[segment.start_row:segment.end_row_exclusive]
    start_ts_ns = timestamps_ns[0]
    times_s = [(timestamp_ns - start_ts_ns) / 1_000_000_000.0 for timestamp_ns in timestamps_ns]
    return times_s, values_xyz


def build_query_times(window_start_s: float) -> List[float]:
    return [window_start_s + (index / TARGET_HZ) for index in range(WINDOW_LENGTH)]


def interpolate_stream(
    times_s: Sequence[float],
    values_xyz: Sequence[Tuple[float, float, float]],
    query_times_s: Sequence[float],
) -> List[List[float]]:
    if len(times_s) != len(values_xyz):
        raise ValueError("times and values must have the same length")
    if len(times_s) < 2:
        raise ValueError("need at least two samples for interpolation")

    axis_values = [
        [xyz[0] for xyz in values_xyz],
        [xyz[1] for xyz in values_xyz],
        [xyz[2] for xyz in values_xyz],
    ]
    output = [[0.0 for _ in query_times_s] for _ in range(3)]
    pointer = 0

    for query_index, query_time_s in enumerate(query_times_s):
        if query_time_s <= times_s[0]:
            left_index = 0
            right_index = 1
        elif query_time_s >= times_s[-1]:
            left_index = len(times_s) - 2
            right_index = len(times_s) - 1
        else:
            while pointer + 1 < len(times_s) and times_s[pointer + 1] < query_time_s:
                pointer += 1
            left_index = pointer
            right_index = pointer + 1

        left_time = times_s[left_index]
        right_time = times_s[right_index]
        if right_time <= left_time:
            alpha = 0.0
        else:
            alpha = (query_time_s - left_time) / (right_time - left_time)
            alpha = min(1.0, max(0.0, alpha))

        for axis in range(3):
            left_value = axis_values[axis][left_index]
            right_value = axis_values[axis][right_index]
            output[axis][query_index] = left_value + alpha * (right_value - left_value)

    return output


def generate_window_centers(common_duration_s: float) -> List[float]:
    half_window = WINDOW_SECONDS / 2.0
    centers: List[float] = []
    center = half_window
    max_center = common_duration_s - half_window
    epsilon = 1e-9
    while center <= max_center + epsilon:
        centers.append(center)
        center += WINDOW_STRIDE_SECONDS
    return centers


def build_segments_by_key(segments_by_stream: Dict[Tuple[str, str], List[SegmentRecord]]) -> Dict[Tuple[str, str], Dict[Tuple[str, int], SegmentRecord]]:
    output: Dict[Tuple[str, str], Dict[Tuple[str, int], SegmentRecord]] = {}
    for stream, segment_list in segments_by_stream.items():
        output[stream] = {(segment.activity_label, segment.occurrence_index): segment for segment in segment_list}
    return output


def generate_windows_for_subject(
    data_root: Path,
    dataset_split: str,
    subject_id: int,
) -> Iterator[Tuple[dict, List[List[float]]]]:
    raw_split = map_dataset_split_to_raw_split(dataset_split)
    subject_streams = load_subject_streams(data_root, raw_split, subject_id)
    segments_by_stream = {
        stream: extract_segments(stream_data)
        for stream, stream_data in subject_streams.items()
    }
    segment_lookup = build_segments_by_key(segments_by_stream)
    paired_segments = pair_segments_for_subject(raw_split, subject_id, segments_by_stream)

    for paired_segment in paired_segments:
        if not paired_segment.is_valid_four_stream_pair:
            continue

        key = (paired_segment.activity_label, paired_segment.occurrence_index)
        stream_payloads = {}
        for stream in REQUIRED_STREAMS:
            segment = segment_lookup[stream][key]
            stream_payloads[stream] = extract_segment_times_and_values(subject_streams[stream], segment)

        centers = generate_window_centers(paired_segment.common_duration_s)
        for offset_index, center_s in enumerate(centers):
            window_start_s = center_s - (WINDOW_SECONDS / 2.0)
            window_end_s = window_start_s + WINDOW_SECONDS
            query_times_s = build_query_times(window_start_s)
            fused_channels: List[List[float]] = []

            for stream in REQUIRED_STREAMS:
                times_s, values_xyz = stream_payloads[stream]
                fused_channels.extend(interpolate_stream(times_s, values_xyz, query_times_s))

            metadata = {
                "split": dataset_split,
                "raw_split": raw_split,
                "subject_id": subject_id,
                "activity_label": paired_segment.activity_label,
                "occurrence_index": paired_segment.occurrence_index,
                "offset_index": offset_index,
                "center_s": center_s,
                "start_s": window_start_s,
                "end_s": window_end_s,
            }
            yield metadata, fused_channels


def compute_label_mapping(subject_splits: Dict[str, List[int]]) -> Dict[str, int]:
    labels = set()
    for dataset_split, subject_ids in subject_splits.items():
        raw_split = map_dataset_split_to_raw_split(dataset_split)
        for subject_id in subject_ids:
            phone_accel_path = DATA_ROOT / raw_split / "phone" / "accel" / f"data_{subject_id}_accel_phone.txt"
            stream = load_stream(phone_accel_path)
            labels.update(stream.labels)
    return {label: index for index, label in enumerate(sorted(labels))}


def accumulate_channel_stats(window_iterator: Iterable[Tuple[dict, List[List[float]]]]) -> Dict[str, List[float]]:
    channel_sums = [0.0] * 12
    channel_sumsq = [0.0] * 12
    channel_counts = [0] * 12
    window_count = 0

    for _, fused_channels in window_iterator:
        window_count += 1
        for channel_index, channel_values in enumerate(fused_channels):
            for value in channel_values:
                channel_sums[channel_index] += value
                channel_sumsq[channel_index] += value * value
                channel_counts[channel_index] += 1

    means: List[float] = []
    stds: List[float] = []
    for channel_index in range(12):
        mean = channel_sums[channel_index] / channel_counts[channel_index]
        variance = (channel_sumsq[channel_index] / channel_counts[channel_index]) - (mean * mean)
        means.append(mean)
        stds.append(math.sqrt(max(variance, 1e-8)))

    return {
        "means": means,
        "stds": stds,
        "window_count": window_count,
    }


def normalize_window(fused_channels: List[List[float]], means: Sequence[float], stds: Sequence[float]) -> List[List[float]]:
    normalized: List[List[float]] = []
    for channel_index, channel_values in enumerate(fused_channels):
        mean = means[channel_index]
        std = stds[channel_index]
        normalized.append([(value - mean) / std for value in channel_values])
    return normalized


class ChunkWriter:
    def __init__(self, root: Path, split: str, chunk_size: int = 256):
        self.root = root / split
        self.root.mkdir(parents=True, exist_ok=True)
        self.split = split
        self.chunk_size = chunk_size
        self.buffer: List[dict] = []
        self.chunk_index = 0
        self.metadata_path = self.root / "metadata.csv"
        self.metadata_handle = self.metadata_path.open("w", encoding="utf-8", newline="")
        self.metadata_writer: Optional[csv.DictWriter] = None

    def add(self, metadata: dict, fused_channels: List[List[float]]) -> None:
        if self.metadata_writer is None:
            fieldnames = list(metadata.keys())
            self.metadata_writer = csv.DictWriter(self.metadata_handle, fieldnames=fieldnames)
            self.metadata_writer.writeheader()
        self.metadata_writer.writerow(metadata)
        self.buffer.append({"window_id": metadata["window_id"], "x_fusion": fused_channels})
        if len(self.buffer) >= self.chunk_size:
            self.flush()

    def flush(self) -> None:
        if not self.buffer:
            return
        chunk_path = self.root / f"data_chunk_{self.chunk_index:04d}.pkl"
        with chunk_path.open("wb") as handle:
            pickle.dump(self.buffer, handle)
        self.buffer = []
        self.chunk_index += 1

    def close(self) -> None:
        self.flush()
        self.metadata_handle.close()


def build_window_cache(data_root: Path, artifact_root: Path, chunk_size: int = 256) -> dict:
    subject_splits = build_dataset_subject_splits(data_root)
    label_to_index = compute_label_mapping(subject_splits)
    window_root = artifact_root / "windows"
    window_root.mkdir(parents=True, exist_ok=True)

    train_stats = accumulate_channel_stats(
        window
        for subject_id in subject_splits["train"]
        for window in generate_windows_for_subject(data_root, "train", subject_id)
    )

    manifest = {
        "target_hz": TARGET_HZ,
        "window_seconds": WINDOW_SECONDS,
        "window_stride_seconds": WINDOW_STRIDE_SECONDS,
        "window_length": WINDOW_LENGTH,
        "label_to_index": label_to_index,
        "subject_splits": subject_splits,
        "normalization": {
            "means": train_stats["means"],
            "stds": train_stats["stds"],
            "train_window_count": train_stats["window_count"],
        },
        "split_window_counts": {},
    }

    window_id = 0
    for dataset_split, subject_ids in subject_splits.items():
        writer = ChunkWriter(window_root, dataset_split, chunk_size=chunk_size)
        split_window_count = 0
        for subject_id in subject_ids:
            for metadata, fused_channels in generate_windows_for_subject(data_root, dataset_split, subject_id):
                normalized_window = normalize_window(fused_channels, train_stats["means"], train_stats["stds"])
                metadata = dict(metadata)
                metadata["window_id"] = window_id
                metadata["label_idx"] = label_to_index[metadata["activity_label"]]
                writer.add(metadata, normalized_window)
                window_id += 1
                split_window_count += 1
        writer.close()
        manifest["split_window_counts"][dataset_split] = split_window_count

    (window_root / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    return manifest


In [ ]:
subject_splits = build_dataset_subject_splits(DATA_ROOT)
print("Dataset subject split:")
print(json.dumps(subject_splits, indent=2))

demo_subject = subject_splits["train"][0]
demo_window = next(generate_windows_for_subject(DATA_ROOT, "train", demo_subject))
demo_metadata, demo_fused_window = demo_window
print("\nDemo window metadata:")
print(json.dumps(demo_metadata, indent=2))
print(f"Fused window shape: ({len(demo_fused_window)}, {len(demo_fused_window[0])})")

# Flip this to True when you want to build the full normalized cache artifacts.
BUILD_FULL_WINDOW_CACHE = False

if BUILD_FULL_WINDOW_CACHE:
    manifest = build_window_cache(DATA_ROOT, ARTIFACT_ROOT, chunk_size=256)
    print("\nWindow cache manifest:")
    print(json.dumps(manifest, indent=2))
else:
    print("\nFull cache build skipped. Set BUILD_FULL_WINDOW_CACHE = True to materialize artifacts.")